# v5等beta价值的状态与普通分钟控制残差

只在2018/2019成熟标签训练两个预定Ridge控制，2020 OOF残差发现。2021/2022不参与特征选择。原生Qlib DatasetH与Recorder保存模型、预测及每日残差。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
root=Path.cwd();card=root/"research/stage1_v5_equal_beta_control_residual_20261030"
if (card/"summary.json").exists():
    import qlib
    from src.core.recorder_sources import load_verified_recorder_objects
    summary=json.loads((card/"summary.json").read_text())
    assert summary["plan_sha256"]==hashlib.sha256((card/"plan.json").read_bytes()).hexdigest()
    qlib.init(provider_uri=str(root/"data/qlib/stage1_event_guard_fullminute_20261028"),region="us",kernels=1)
    saved,receipt=load_verified_recorder_objects(card,expected_recorder_id=summary["recorder_id"],
        artifacts=["models/state_only","models/ordinary_minute","analysis/predictions","analysis/summary"],
        expected_manifest_sha256=hashlib.sha256((card/"verification_manifest.json").read_bytes()).hexdigest())
    assert len(saved["analysis/predictions"])==summary["evaluation_rows"]
    print("verified_existing_control_recorder",summary["recorder_id"],summary["evaluation_days"])
else:
    import os
    os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
    os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
    import hashlib,json
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import qlib
    from qlib.data import D
    from qlib.data.dataset import DatasetH
    from qlib.data.dataset.handler import DataHandlerLP
    from qlib.workflow import R
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from src.core.recorder_sources import load_verified_recorder_objects
    from src.workflow.minute_action_value import _source
    root=Path.cwd();card=root/"research/stage1_v5_equal_beta_control_residual_20261030"
    plan_bytes=(card/"plan.json").read_bytes();plan=json.loads(plan_bytes);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
    label_card=root/plan["label_source"]["card"]
    assert sha(label_card/"summary.json")==plan["label_source"]["summary_sha256"]
    assert sha(label_card/"verification_manifest.json")==plan["label_source"]["manifest_sha256"]
    assert json.loads((label_card/"nas_binding.json").read_text())["evidence"]["sha256"]==plan["label_source"]["nas_archive_sha256"]
    assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
    qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
    label_objects,label_receipt=load_verified_recorder_objects(label_card,expected_recorder_id=plan["label_source"]["recorder_id"],
        artifacts=["analysis/labels","analysis/audit"],expected_manifest_sha256=plan["label_source"]["manifest_sha256"])
    labels=label_objects["analysis/labels"]
    assert len(labels)==label_objects["analysis/audit"]["event_slots"]
    labels=labels.loc[labels.trade_date.dt.year.le(2020) & labels.soxl_funded & labels.soxx_funded].copy()
    source,source_receipt=_source(root,plan["training_source"],["analysis/native/dated_f696_0"])
    frame_source,frame_receipt=_source(root,plan["frame_source"],["inputs/covered_frame"])
    book=source["analysis/native/dated_f696_0"]
    frame=frame_source["inputs/covered_frame"].reset_index(drop=True)
    assert frame.trade_date.equals(book["authority"].trade_date)
    frame=frame.loc[frame.trade_date.dt.year.le(2020)].copy()
    assert list(frame.columns[3:42])==plan["base39_features"]
    assert frame.feature_available_at.lt(frame.trade_date+pd.Timedelta(hours=9,minutes=46)).all()
    state=frame[["trade_date",*plan["base39_features"],"soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]].copy()
    state["risk_state"]=book["authority"].loc[book["authority"].trade_date.dt.year.le(2020),"risk_state"].to_numpy()
    state["risk_clock"]=book["authority"].loc[book["authority"].trade_date.dt.year.le(2020),"risk_clock"].to_numpy()
    quant=[]
    for day in state.trade_date:
     stamp=pd.Timestamp(day)+pd.Timedelta(hours=9,minutes=47)
     pos=book["native_positions"].get(stamp)
     assert pos is not None
     quant.append([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ("SOXL","SOXX")])
    state[["q_soxl","q_soxx"]]=np.asarray(quant,float)
    paired=labels.merge(state,on="trade_date",how="left",validate="many_to_one")
    assert paired[plan["base39_features"]].notna().all().all()
    bars=D.features(["SOXL","SOXX"],["$close","$raw_volume","$price_valid"],start_time="2018-10-15 09:31:00",end_time="2020-12-31 16:00:00",freq="1min")
    parts=[]
    for symbol in ("SOXL","SOXX"):
     part=bars.xs(symbol,level="instrument").copy()
     part=part.loc[part.index.normalize().isin(pd.DatetimeIndex(paired.trade_date.unique()))]
     assert part["$price_valid"].eq(1).all()
     days=part.index.normalize();logp=np.log(part["$close"].astype(float))
     out=pd.DataFrame(index=part.index)
     for width in (5,15):out[f"minute_{symbol.lower()}_ret{width}"]=logp-logp.groupby(days).shift(width)
     out[f"minute_{symbol.lower()}_log1p_raw_volume"]=np.log1p(part["$raw_volume"].astype(float))
     out[f"{symbol.lower()}_mark"]=part["$close"].astype(float)
     parts.append(out)
    minute=parts[0].join(parts[1],how="inner")
    paired=paired.join(minute,on="bar_end_at",how="left")
    assert paired[["soxl_mark","soxx_mark"]].notna().all().all()
    paired["decision_soxl_weight"]=paired.q_soxl*paired.soxl_mark/paired.decision_nav
    paired["decision_soxx_weight"]=paired.q_soxx*paired.soxx_mark/paired.decision_nav
    paired["clock_fraction"]=(paired.bar_end_at.dt.hour*60+paired.bar_end_at.dt.minute-(9*60+30))/390.
    assert paired.label_available_at.gt(paired.decision_at).all()
    assert paired.trade_date.dt.year.isin([2018,2019,2020]).all()
    state_features=plan["base39_features"]+plan["state_features"]
    full_features=state_features+plan["ordinary_minute_features"]
    assert len(set(full_features))==len(full_features)
    packed=pd.concat({"feature":paired[full_features],"label":paired[[plan["primary_label"]]],
        "metadata":paired[["trade_date","decision_at","label_available_at"]]},axis=1)
    packed.index=pd.MultiIndex.from_arrays([paired.decision_at,["SOXL_SOXX_CASH"]*len(paired)],names=["datetime","instrument"])
    dataset=DatasetH(handler=DataHandlerLP.from_df(packed),segments={"train":("2018-10-15","2019-12-31"),"test":("2020-01-01","2020-12-31")})
    train=dataset.prepare("train",col_set=["feature","label","metadata"])
    test=dataset.prepare("test",col_set=["feature","label","metadata"])
    first_test=pd.Timestamp(test.index.get_level_values("datetime").min())
    train=train.loc[pd.to_datetime(train[("metadata","label_available_at")]).lt(first_test)].copy()
    assert train.index.get_level_values("datetime").max()<first_test
    assert train[("metadata","trade_date")].dt.year.isin([2018,2019]).all()
    assert test[("metadata","trade_date")].dt.year.eq(2020).all()
    ytrain=train[("label",plan["primary_label"])].to_numpy(float)
    ytest=test[("label",plan["primary_label"])].to_numpy(float)
    train_days=pd.to_datetime(train[("metadata","trade_date")]).to_numpy()
    test_days=pd.to_datetime(test[("metadata","trade_date")]).to_numpy()
    train_n=pd.Series(train_days).map(pd.Series(train_days).value_counts()).to_numpy(float)
    weights=1./train_n
    models={};predictions={};metrics={}
    for name,features in (("state_only",state_features),("ordinary_minute",full_features)):
     model=make_pipeline(SimpleImputer(strategy="median",add_indicator=True),StandardScaler(),Ridge(alpha=1.0))
     model.fit(train.loc[:,pd.IndexSlice["feature",features]],ytrain,ridge__sample_weight=weights)
     pred=model.predict(test.loc[:,pd.IndexSlice["feature",features]])
     models[name]=model;predictions[name]=pred
     perday=pd.DataFrame({"day":test_days,"squared_error":(ytest-pred)**2}).groupby("day").squared_error.mean()
     metrics[name]={"test_days":len(perday),"test_rows":len(test),"day_weighted_mse":float(perday.mean()),"mean_prediction":float(np.mean(pred))}
    residual=ytest-predictions["ordinary_minute"]
    diagnostic=pd.DataFrame({"trade_date":test_days,"decision_at":test.index.get_level_values("datetime"),"actual_value":ytest,
        "state_prediction":predictions["state_only"],"ordinary_prediction":predictions["ordinary_minute"],"ordinary_residual":residual})
    daily=diagnostic.groupby("trade_date").ordinary_residual.mean()
    centered=diagnostic.ordinary_residual-diagnostic.trade_date.map(daily)
    variance={"total":float(np.var(residual)),"between_day":float(np.var(diagnostic.trade_date.map(daily))),
              "within_day":float(np.var(centered)),"within_day_share":float(np.var(centered)/np.var(residual))}
    with R.start(experiment_name=card.name,uri=str(card/"qlib_recorder")):
     rec=R.get_recorder();rec.save_objects(artifact_path="inputs",plan=plan,source_readbacks={"labels":label_receipt,"parent":source_receipt,"frame":frame_receipt})
     rec.save_objects(artifact_path="models",state_only=models["state_only"],ordinary_minute=models["ordinary_minute"])
     rec.save_objects(artifact_path="analysis",predictions=diagnostic,daily_residual=daily,metrics=metrics,variance=variance)
     rec.log_metrics(state_day_mse=metrics["state_only"]["day_weighted_mse"],ordinary_day_mse=metrics["ordinary_minute"]["day_weighted_mse"],within_day_residual_share=variance["within_day_share"])
    summary={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"decision":"descriptive_2020_oof_control_residual_no_candidate_feature",
     "training_rows":len(train),"training_days":int(pd.Series(train_days).nunique()),"evaluation_rows":len(test),"evaluation_days":int(pd.Series(test_days).nunique()),
     "metrics":metrics,"residual_variance":variance,"new_model_fits":2,"new_native_accounts":0,"historical_research_passed":False,
     "recorder_id":rec.id,"experiment_id":str(rec.experiment_id),"recorder_uri":str(card/"qlib_recorder"),"recorder_local_dir":str(card/"qlib_recorder"/str(rec.experiment_id)/rec.id),
     "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["inputs/*","models/*","analysis/*"]},
     "limits":["2020 residuals are discovery evidence, not a candidate feature or independent forward test.","Funded-pair conditioning uses future fill prices and is not a live-intent policy.","2021/2022 label outcomes were not evaluated."]}
    (card/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n")
    print(json.dumps({"decision":summary["decision"],"train_days":summary["training_days"],"eval_days":summary["evaluation_days"],"metrics":metrics,"variance":variance},ensure_ascii=False))

## 中文结论

2018/2019成熟304日109795行训练两个固定Ridge控制；2020 OOF242日86676行。状态MSE1.803317e-8，增加普通分钟涨跌/量仅降至1.798565e-8（约0.26%）；事后只读训练均值常数MSE1.240507e-8，表明两个控制自身跨年失准。2020四季度真实SOXL−SOXX费后价值均略负，控制预测均正；剩余误差约71%在日间、29%日内。该常数对照不是冻结候选门；没有新特征、完整账户或SOXX超额。2021/2022标签未参与训练或评估。模型、预测、每日残差在Recorder，季度与训练标签日均及常数诊断在sidecar。NAS上传与选定工件远端读回通过。下一步先用2018/2019内部信息确定稳健控制，再只在2020找残差，不按2020调旧Ridge；事后填单双臂筛样本不是实时意向策略。